# Task 4 (Part B): YOLO Object Detector, Frame-by-Frame Pipeline
Pre-trained YOLO model + tracking on a video, with a dynamic class-count overlay and frame-by-frame inference metrics.

**Pehle ye karo:** `Runtime > Change runtime type > T4 GPU`, phir har cell ko upar se neeche **Shift + Enter** se chalao.
Aapko ek **chhoti video** (10 se 30 second) chahiye hogi: phone se koi bhi video record karo (sarak, kamra, desk par cheezein). Cell 3 me wo upload hogi.

## Cell 1: Install and imports

In [ ]:
!pip -q install ultralytics
import time, json, os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from collections import Counter
from ultralytics import YOLO

print("GPU available:", torch.cuda.is_available())

## Cell 2: Settings

In [ ]:
MODEL_NAME = "yolov8n.pt"   # small and fast. Other options: "yolov8s.pt" (more accurate), "yolo11n.pt"
CONF       = 0.35           # minimum confidence for a detection
IMG_SIZE   = 640            # inference resolution
USE_TRACKING = True         # give every object a persistent ID (ByteTrack)
MAX_FRAMES = 900            # safety limit (the room video has 650 frames)
OUT_DIR    = "yolo_results"
os.makedirs(OUT_DIR, exist_ok=True)

## Cell 3: Video upload karo

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO = list(uploaded.keys())[0]
print("Using video:", VIDEO)

## Cell 4: Model load

In [ ]:
model = YOLO(MODEL_NAME)
print("Classes the model knows:", len(model.names))
# warm-up so that the first frame is not slower because of GPU start-up
_ = model(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8), verbose=False)

## Cell 5: Frame-by-frame inference (main loop)

In [ ]:
cap = cv2.VideoCapture(VIDEO)
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
FPS_IN = cap.get(cv2.CAP_PROP_FPS) or 30
writer = cv2.VideoWriter(f"{OUT_DIR}/yolo_annotated.mp4", cv2.VideoWriter_fourcc(*"mp4v"), FPS_IN, (W, H))

rng = np.random.default_rng(1)
palette = {}
def color_of(name):
    if name not in palette:
        palette[name] = tuple(int(x) for x in rng.integers(60, 255, 3))
    return palette[name]

rows, all_conf, class_total = [], [], Counter()
sample_frames = {}
f = 0
while f < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()
    try:
        res = (model.track(frame, persist=True, conf=CONF, imgsz=IMG_SIZE, verbose=False)[0]
               if USE_TRACKING else model(frame, conf=CONF, imgsz=IMG_SIZE, verbose=False)[0])
    except Exception as e:                       # tracking needs an extra package; fall back to plain detection
        print("Tracking not available, using plain detection:", str(e)[:80])
        USE_TRACKING = False
        res = model(frame, conf=CONF, imgsz=IMG_SIZE, verbose=False)[0]
    latency = (time.perf_counter() - t0) * 1000

    boxes = res.boxes
    names = [model.names[int(k)] for k in boxes.cls.tolist()] if len(boxes) else []
    confs = boxes.conf.tolist() if len(boxes) else []
    ids = boxes.id.int().tolist() if (len(boxes) and boxes.id is not None) else [None] * len(names)
    counts = Counter(names)
    class_total.update(names)
    all_conf += confs

    out = frame.copy()
    for (x1, y1, x2, y2), n, cf, tid in zip(boxes.xyxy.tolist() if len(boxes) else [], names, confs, ids):
        col = color_of(n)
        cv2.rectangle(out, (int(x1), int(y1)), (int(x2), int(y2)), col, 2)
        label = f"{'#' + str(tid) + ' ' if tid is not None else ''}{n} {cf:.2f}"
        cv2.putText(out, label, (int(x1), max(14, int(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 2, cv2.LINE_AA)
    # HUD with the dynamic count vector (top classes in this frame)
    top = counts.most_common(5)
    hud_h = 30 + 22 * (len(top) + 1)
    panel = out.copy(); cv2.rectangle(panel, (0, 0), (330, hud_h), (0, 0, 0), -1)
    out = cv2.addWeighted(panel, 0.55, out, 0.45, 0)
    cv2.putText(out, f"objects in frame: {len(names)}", (8, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 255), 1, cv2.LINE_AA)
    for i, (n, k) in enumerate(top):
        cv2.putText(out, f"{n}: {k}", (8, 44 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(out, f"FPS {1000/latency:5.1f}  latency {latency:5.1f} ms", (8, 44 + 22 * len(top)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 1, cv2.LINE_AA)
    writer.write(out)

    rows.append(dict(frame=f, latency_ms=latency, preprocess_ms=res.speed["preprocess"], inference_ms=res.speed["inference"],
                     postprocess_ms=res.speed["postprocess"], n_objects=len(names),
                     mean_conf=float(np.mean(confs)) if confs else 0.0, counts=json.dumps(dict(counts))))
    if f in (10, 40, 80, 120):
        sample_frames[f] = out
    f += 1

cap.release(); writer.release()
df = pd.DataFrame(rows)
df.to_csv(f"{OUT_DIR}/yolo_frame_metrics.csv", index=False)
print(f"Processed {len(df)} frames ({W}x{H}).")

## Cell 6: Summary metrics + figures

In [ ]:
lat = df["latency_ms"]
changes = df["n_objects"].diff().abs().dropna()
summary = dict(
    model=MODEL_NAME, conf_threshold=CONF, imgsz=IMG_SIZE, tracking=USE_TRACKING, frames=len(df), resolution=f"{W}x{H}",
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    latency_ms=dict(mean=lat.mean(), median=lat.median(), p95=lat.quantile(0.95), max=lat.max(), std=lat.std()),
    fps_mean=1000 / lat.mean(),
    stage_mean_ms=dict(preprocess=df["preprocess_ms"].mean(), inference=df["inference_ms"].mean(), postprocess=df["postprocess_ms"].mean()),
    objects_per_frame_mean=df["n_objects"].mean(), objects_per_frame_max=int(df["n_objects"].max()),
    mean_confidence=float(np.mean(all_conf)) if all_conf else 0.0,
    count_flicker_mean_abs_change=float(changes.mean()) if len(changes) else 0.0,
    class_detections_total=dict(class_total.most_common()),
)
json.dump(summary, open(f"{OUT_DIR}/yolo_summary.json", "w"), indent=2, default=float)
print(json.dumps(summary, indent=2, default=float))

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0, 0].plot(df["frame"], lat, lw=0.8); ax[0, 0].axhline(lat.mean(), color="r", ls="--", label="mean")
ax[0, 0].set(title="Per-frame latency", xlabel="Frame", ylabel="ms"); ax[0, 0].legend()
ax[0, 1].plot(df["frame"], df["n_objects"]); ax[0, 1].set(title="Objects detected per frame", xlabel="Frame", ylabel="Count")
ax[1, 0].hist(all_conf, bins=20, color="tab:green"); ax[1, 0].set(title="Confidence distribution", xlabel="Confidence", ylabel="Detections")
top_cls = class_total.most_common(8)
ax[1, 1].barh([k for k, _ in top_cls][::-1], [v for _, v in top_cls][::-1], color="tab:purple")
ax[1, 1].set(title="Most detected classes (all frames)", xlabel="Detections")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/yolo_metrics.png", dpi=130); plt.show()

for k, im in sample_frames.items():
    cv2.imwrite(f"{OUT_DIR}/yolo_frame_{k:04d}.png", im)

## Cell 7: Sample frames dekho

In [ ]:
fig, ax = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 4))
ax = np.atleast_1d(ax)
for a, (k, im) in zip(ax, sample_frames.items()):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(f"Frame {k}"); a.axis("off")
plt.tight_layout(); plt.show()

## Cell 8: Sab results ek zip me download karo

In [ ]:
!zip -qr yolo_results.zip yolo_results
files.download("yolo_results.zip")